<a href="https://colab.research.google.com/github/samrat-dhakal-11/Google-Colab-Notebooks/blob/main/mcp_composio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "composio>=0.17.1" "langchain[mcp]>=1.4,<2" langchain-openai
!pip install composio langchain langchain-openai langchain-mcp-adapters python-dotenv

In [2]:
from google.colab import userdata
ckey=userdata.get("COMPOSIO_API_KEY")
cuser_id=userdata.get("COMPOSIO_USER_ID")
Key=userdata.get("OPENAI_API_KEY")

In [3]:

import asyncio
import os

from composio import Composio
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

In [4]:
composio=Composio(api_key=ckey)

from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-5.4-mini",api_key=Key)

In [5]:
import asyncio

async def main():
  session=composio.create(user_id=cuser_id)
  client=MultiServerMCPClient({
    'composio':{"transport":"streamable_http","url":session.mcp.url,"headers":session.mcp.headers}
  })
  tools= await client.get_tools()
  agent=create_agent(llm,tools)
  response= await agent.ainvoke({'messages':[("user",query)]})
  print(response['messages'][-1].content)

In [ ]:
while(True):
  query=input("Enter your question: ")
  if query.strip().lower() in ["end","exit","bye","cancel"]:
    break
  print("\n","-"*120)
  print(query,": ",await main())

In [ ]:
query="""
Search every platform  you connected to and return my user name
"""

In [ ]:

import os
import asyncio

from composio import Composio
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

TOOLKITS = [
    "gmail", "github", "googlecalendar", "notion", "googlesheets",
    "slack", "googledrive", "googledocs", "youtube", "linkedin",
    "googlemeet", "googlephotos", "facebook", "instagram",
]

async def main():
    composio = Composio(api_key=ckey)
    session = composio.create(
        user_id=cuser_id,
        toolkits=TOOLKITS,
    )

    client = MultiServerMCPClient({
        "composio": {
            "transport": "streamable_http",
            "url": session.mcp.url,
            "headers": session.mcp.headers,
        }
    })

    tools = await client.get_tools()
    print("Loaded tools length :", len(tools))

    agent = create_agent(llm, tools)

    result = await agent.ainvoke({
        "messages": [
            ("user", query)
        ]
    })
    print(result["messages"][-1].content)

await main()